In [5]:
# CELL 1 — Install & Clone
import os, functools
print = functools.partial(print, flush=True)

os.system("pip install -q timm==1.0.15 open_clip_torch==2.26.1 albumentations scikit-learn pandas tqdm")
print("Packages installed ✅")

if not os.path.exists("ALD-E-ImageMiner"):
    ret = os.system("git clone --depth 1 https://github.com/sciknoworg/ALD-E-ImageMiner.git")
    print("Cloned ✅" if ret == 0 else "❌ Clone failed")
else:
    print("Repo already exists ✅")

DATA_ROOT  = "ALD-E-ImageMiner/icdar2026-competition-data"
TRAIN_ROOT = DATA_ROOT + "/train"
DEV_ROOT   = DATA_ROOT + "/dev"
TEST_ROOT  = DATA_ROOT + "/test"

Packages installed ✅
Cloned ✅


In [6]:
# CELL 2 — Imports
import os, glob, json, cv2, warnings, random, gc, zipfile, re
import numpy as np
import pandas as pd
from tqdm import tqdm
from collections import Counter, defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import GradScaler, autocast

import open_clip
import albumentations as A
from albumentations.pytorch import ToTensorV2
from PIL import Image as PILImage

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Device: cuda
GPU: Tesla T4


In [ ]:
# CELL 3 — Config
CFG = dict(
    CLIP_MODEL      = "ViT-L-14-336",
    CLIP_PRETRAINED = "openai",
    IMG_SIZE        = 336,
    BATCH_SIZE      = 16,         
    EPOCHS          = 25,         
    LR              = 3e-5,
    MIN_LR          = 1e-7,
    WARMUP_EPOCHS   = 2,           
    WEIGHT_DECAY    = 5e-3,
    UNFREEZE_BLOCKS = 12,          
    DROPOUT         = 0.30,       
    FOCAL_GAMMA     = 2.0,
    LABEL_SMOOTH    = 0.08,        
    PATIENCE        = 8,           
    SEED            = 42,
    NUM_WORKERS     = 2,
    SAVE_PATH       = "model_stage1.pth",
    # Stage 2
    FT_EPOCHS       = 15,          
    FT_LR           = 1e-5,        
    FT_PATIENCE     = 6,           
    FT_UNFREEZE_ALL = True,       
    SAVE_PATH_FULL  = "model_stage2.pth",
    # Pseudo-label
    PSEUDO_THRESH   = 0.85,        
    # MixUp / CutMix
    MIXUP_ALPHA     = 0.4,         # MixUp interpolation strength
    CUTMIX_ALPHA    = 1.0,         # CutMix Beta distribution alpha
    MIXUP_PROB      = 0.5,         # prob of applying MixUp or CutMix per batch
    # Caption fusion
    USE_CAPTION     = True,        # fuse CLIP text embeddings from captions
    CAPTION_WEIGHT  = 0.3,         # weight of text branch in fusion
)

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(CFG["SEED"])
print("Config ready ✅")

Config ready ✅


In [ ]:
# CELL 4 — Data Extraction 
train_jsons = glob.glob(TRAIN_ROOT + "/**/images/*.json", recursive=True)
dev_jsons   = glob.glob(DEV_ROOT   + "/**/images/*.json", recursive=True)
test_jsons  = glob.glob(TEST_ROOT  + "/**/images/*.json", recursive=True)
print(f"JSON files — Train: {len(train_jsons)} | Dev: {len(dev_jsons)} | Test: {len(test_jsons)}")


def extract_panels(json_files, save_dir, is_test=False):
    os.makedirs(save_dir, exist_ok=True)
    records, skipped = [], 0
    for jp in tqdm(json_files, desc=f"Extracting → {save_dir}"):
        try:
            with open(jp) as f:
                data = json.load(f)
        except Exception:
            skipped += 1; continue
        if "bbox" not in data:
            skipped += 1; continue
        img_path = jp.replace(".json", ".jpg")
        if not os.path.exists(img_path):
            img_path = jp.replace(".json", ".png")
        if not os.path.exists(img_path):
            skipped += 1; continue
        img = cv2.imread(img_path)
        if img is None:
            skipped += 1; continue
        H, W      = img.shape[:2]
        sample_id = data.get("sample_id", "")
        caption   = data.get("caption", "") or ""
        panel_keys = data["bbox"].keys() if is_test else data.get("classification", {}).keys()
        for panel in panel_keys:
            if panel not in data["bbox"]: continue
            bbox = data["bbox"][panel]
            x  = max(0, int(bbox["x"]))
            y  = max(0, int(bbox["y"]))
            x2 = min(W, x + int(bbox["width"]))
            y2 = min(H, y + int(bbox["height"]))
            crop = img[y:y2, x:x2]
            if crop.size == 0: crop = img
            fname = sample_id.replace("/", "_") + "_" + panel + ".jpg"
            fpath = os.path.join(save_dir, fname)
            cv2.imwrite(fpath, crop)
            rec = dict(image=fpath, sample_id=sample_id, panel=panel, caption=caption)
            if not is_test:
                rec["label"] = data.get("classification", {}).get(panel, "unknown")
            records.append(rec)
    print(f"  → {len(records)} panels extracted, {skipped} skipped")
    return pd.DataFrame(records)


CACHE_TRAIN = "train_panels.csv"
CACHE_DEV   = "dev_panels.csv"
CACHE_TEST  = "test_panels.csv"

if os.path.exists(CACHE_TRAIN) and os.path.exists(CACHE_DEV) and os.path.exists(CACHE_TEST):
    train_df = pd.read_csv(CACHE_TRAIN)
    dev_df   = pd.read_csv(CACHE_DEV)
    test_df  = pd.read_csv(CACHE_TEST)
    print("Loaded from cache ✅")
else:
    train_df = extract_panels(train_jsons, "panels_train")
    dev_df   = extract_panels(dev_jsons,   "panels_dev")
    test_df  = extract_panels(test_jsons,  "panels_test", is_test=True)
    train_df.to_csv(CACHE_TRAIN, index=False)
    dev_df.to_csv(CACHE_DEV,    index=False)
    test_df.to_csv(CACHE_TEST,  index=False)
    print("Saved to CSV ✅")

for df in [train_df, dev_df, test_df]:
    df["caption"] = df["caption"].fillna("")

print(f"Train: {len(train_df)} | Dev: {len(dev_df)} | Test: {len(test_df)}")

JSON files — Train: 1170 | Dev: 201 | Test: 580


Extracting → panels_train: 100%|██████████| 1170/1170 [00:05<00:00, 232.98it/s]

  → 2422 panels extracted, 0 skipped



Extracting → panels_dev: 100%|██████████| 201/201 [00:00<00:00, 265.92it/s]

  → 362 panels extracted, 0 skipped



Extracting → panels_test: 100%|██████████| 580/580 [00:02<00:00, 226.31it/s]

  → 1121 panels extracted, 0 skipped


Saved to CSV ✅
Train: 2422 | Dev: 362 | Test: 1121


In [ ]:
# CELL 5 — Label Encoding & Class Weights 
le = LabelEncoder()
le.fit(train_df["label"].values)
train_df["label_id"] = le.transform(train_df["label"])

dev_df_valid = dev_df[dev_df["label"].isin(le.classes_)].copy()
dev_df_valid["label_id"] = le.transform(dev_df_valid["label"])

num_classes = len(le.classes_)
print(f"Num classes : {num_classes}")
print(f"Train rows  : {len(train_df)}")
print(f"Dev rows    : {len(dev_df_valid)}")

class_counts = np.bincount(train_df["label_id"].values, minlength=num_classes)
print(f"\nClass distribution (sorted by count):")
for cid in np.argsort(class_counts):
    print(f"  [{class_counts[cid]:4d}]  {le.classes_[cid]}")

class_weights = compute_class_weight("balanced", classes=np.arange(num_classes),
                                     y=train_df["label_id"].values)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
print(f"\nClass weight range: [{class_weights.min():.3f}, {class_weights.max():.3f}]")

Num classes : 36
Train rows  : 2422
Dev rows    : 362

Class distribution (sorted by count):
  [   1]  area chart
  [   1]  chromaticity diagram
  [   1]  periodic table map
  [   1]  timeline chart
  [   2]  polar chart (rose chart)
  [   2]  formula
  [   3]  table
  [   4]  pie chart
  [   4]  phase diagram
  [   5]  band diagram
  [   6]  box plot
  [   6]  workflow diagram
  [   7]  stacked bar chart
  [   9]  grouped bar chart
  [  10]  device structure diagram
  [  10]  3d scatter plot
  [  16]  process timing diagram
  [  29]  unknown
  [  35]  bar chart
  [  38]  contour heatmap
  [  43]  process flow diagram
  [  51]  apparatus diagram
  [  61]  reaction energy profile diagram
  [  73]  spectra chart
  [  74]  heatmap
  [  82]  multi-axis chart
  [ 114]  stacked spectra chart
  [ 114]  line chart
  [ 130]  reaction scheme
  [ 133]  scatter plot
  [ 145]  conceptual diagram
  [ 152]  multi spectra chart
  [ 176]  multiple scatter plot
  [ 235]  image panel
  [ 262]  multiple l

In [ ]:
# CELL 6 — Dataset, Transforms, Losses, MixUp/CutMix, Utilities

# ── Combined Loss  ───────────────────────────────────────────────
class CombinedLoss(nn.Module):
    def __init__(self, gamma=2.0, smoothing=0.08, weight=None, focal_w=0.7):
        super().__init__()
        self.gamma   = gamma
        self.smooth  = smoothing
        self.weight  = weight
        self.focal_w = focal_w
        self.ce_w    = 1.0 - focal_w

    def focal_loss(self, logits, targets):
        log_p = F.log_softmax(logits, dim=-1)
        p     = torch.exp(log_p)
        nll   = F.nll_loss(log_p, targets, weight=self.weight, reduction="none")
        pt    = p.gather(1, targets.unsqueeze(1)).squeeze(1)
        return ((1 - pt) ** self.gamma * nll).mean()

    def smooth_ce(self, logits, targets):
        n_cls = logits.size(1)
        log_p = F.log_softmax(logits, dim=-1)
        with torch.no_grad():
            smooth_target = torch.full_like(log_p, self.smooth / (n_cls - 1))
            smooth_target.scatter_(1, targets.unsqueeze(1), 1.0 - self.smooth)
        loss = -(smooth_target * log_p).sum(dim=-1)
        if self.weight is not None:
            w = self.weight[targets]
            loss = (loss * w).sum() / w.sum()
        else:
            loss = loss.mean()
        return loss

    def forward(self, logits, targets):
        return self.focal_w * self.focal_loss(logits, targets) + \
               self.ce_w   * self.smooth_ce(logits, targets)


# ── NEW: Soft-target Combined Loss (for MixUp/CutMix mixed labels) ───────────
class SoftCombinedLoss(nn.Module):
    """Handles soft (mixed) targets from MixUp/CutMix."""
    def __init__(self, gamma=2.0, weight=None):
        super().__init__()
        self.gamma  = gamma
        self.weight = weight

    def forward(self, logits, soft_targets):
        # soft_targets: (B, C) probability distribution
        log_p = F.log_softmax(logits, dim=-1)
        p     = torch.exp(log_p)
        # Focal weight: use argmax class probability as pt
        pt    = (p * soft_targets).sum(dim=1)  # expected probability
        focal_weight = (1 - pt) ** self.gamma
        ce    = -(soft_targets * log_p).sum(dim=1)
        return (focal_weight * ce).mean()


# ── NEW: MixUp / CutMix ───────────────────────────────────────────────────────
def mixup_data(x, y_onehot, alpha=0.4):
    """Returns mixed inputs and mixed soft labels."""
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    mixed_x = lam * x + (1 - lam) * x[idx]
    mixed_y = lam * y_onehot + (1 - lam) * y_onehot[idx]
    return mixed_x, mixed_y


def cutmix_data(x, y_onehot, alpha=1.0):
    """Returns CutMix inputs and soft labels."""
    lam = np.random.beta(alpha, alpha)
    B, C, H, W = x.shape
    idx = torch.randperm(B, device=x.device)

    cut_rat = np.sqrt(1.0 - lam)
    cut_h = int(H * cut_rat)
    cut_w = int(W * cut_rat)

    cx = np.random.randint(W)
    cy = np.random.randint(H)
    x1 = max(cx - cut_w // 2, 0)
    x2 = min(cx + cut_w // 2, W)
    y1 = max(cy - cut_h // 2, 0)
    y2 = min(cy + cut_h // 2, H)

    mixed_x = x.clone()
    mixed_x[:, :, y1:y2, x1:x2] = x[idx, :, y1:y2, x1:x2]

    lam_actual = 1 - (x2 - x1) * (y2 - y1) / (W * H)
    mixed_y = lam_actual * y_onehot + (1 - lam_actual) * y_onehot[idx]
    return mixed_x, mixed_y


# ── EarlyStopping — track ACCURACY ──────────────────────────────────────────
class EarlyStopping:
    def __init__(self, patience=8, save_path="best.pth", min_delta=5e-4):
        self.patience  = patience
        self.save_path = save_path
        self.min_delta = min_delta
        self.best_acc  = 0.0
        self.best_f1   = 0.0
        self.counter   = 0
        self.stop      = False

    def step(self, val_acc, val_f1, model):
        if val_acc > self.best_acc + self.min_delta:
            self.best_acc = val_acc
            self.best_f1  = val_f1
            self.counter  = 0
            torch.save(model.state_dict(), self.save_path)
            print(f"  ✅ Best saved (Acc={self.best_acc:.4f}, F1={self.best_f1:.4f})")
        else:
            self.counter += 1
            print(f"  ⏳ No improvement ({self.counter}/{self.patience})  "
                  f"best_acc={self.best_acc:.4f}")
            if self.counter >= self.patience:
                print("  🛑 Early stopping triggered")
                self.stop = True
        return self.stop


# ── Transforms — same as v14 (VerticalFlip already removed) ──────────────────
def get_transforms(img_size, mode="train"):
    norm = dict(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
    if mode == "train":
        return A.Compose([
            A.Resize(img_size, img_size),
            A.HorizontalFlip(p=0.5),
            A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=10, p=0.4),
            A.OneOf([
                A.GaussianBlur(blur_limit=(3, 5), p=1),
                A.Sharpen(p=1),
                A.MedianBlur(blur_limit=3, p=1)
            ], p=0.3),
            A.RandomBrightnessContrast(0.2, 0.2, p=0.4),
            A.HueSaturationValue(15, 25, 15, p=0.3),
            A.CLAHE(p=0.35),
            A.CoarseDropout(max_holes=4, max_height=24, max_width=24, p=0.2),
            A.Normalize(**norm), ToTensorV2()
        ])
    return A.Compose([A.Resize(img_size, img_size), A.Normalize(**norm), ToTensorV2()])


# ── Dataset — now also returns caption string ─────────────────────────────────
class PanelDataset(Dataset):
    def __init__(self, df, transform, has_label=True):
        self.df        = df.reset_index(drop=True)
        self.transform = transform
        self.has_label = has_label

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = cv2.imread(row.image)
        if img is None:
            img = np.zeros((224, 224, 3), dtype=np.uint8)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = self.transform(image=img)["image"]
        caption = str(row.caption) if hasattr(row, "caption") and row.caption else ""
        if self.has_label:
            return img, int(row.label_id), caption
        return img, caption


# ── Weighted Random Sampler ──────────────────────────────────────────────────
def make_weighted_sampler(df, class_weights_arr):
    sample_weights = np.array([class_weights_arr[lid] for lid in df["label_id"].values])
    sample_weights = torch.from_numpy(sample_weights).float()
    return WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True
    )

print("Dataset & utilities defined ✅")

Dataset & utilities defined ✅


In [11]:
# CELL 7 — NEW: CLIP Image+Text Fusion Classifier
# When a figure caption is available, we encode it with CLIP's text encoder
# and fuse it with the visual embedding before the classification head.
# This gives the model a free signal: captions often mention chart type explicitly
# (e.g. "Fig. 3: Scatter plot of deposition rate vs. temperature")

class CLIPFusionClassifier(nn.Module):
    def __init__(self, clip_model, num_classes, dropout=0.30, caption_weight=0.3):
        super().__init__()
        self.clip_model     = clip_model
        self.caption_weight = caption_weight
        embed_dim = clip_model.visual.output_dim

        # Image head
        self.img_head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, 512),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        # Text head (same dim)
        self.txt_head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, 512),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        # Fusion classifier
        self.classifier = nn.Sequential(
            nn.Linear(512, 256),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(256, num_classes),
        )

        # Gating: learnable scalar to weight text contribution
        self.text_gate = nn.Parameter(torch.tensor(caption_weight))

    def encode_text_batch(self, captions):
        """Tokenize and encode a list of caption strings."""
        # open_clip 2.26.1 removed the 'truncate' kwarg from tokenize().
        # Use get_tokenizer (preferred) with a try/except fallback.
        dev = next(self.clip_model.parameters()).device
        try:
            tokenizer = open_clip.get_tokenizer(CFG["CLIP_MODEL"])
            tokens = tokenizer(captions).to(dev)
        except Exception:
            tokens = open_clip.tokenize(captions).to(dev)
        with torch.no_grad():
            txt_f = self.clip_model.encode_text(tokens).float()
        return F.normalize(txt_f, dim=-1)

    def forward(self, images, captions=None):
        img_f = F.normalize(self.clip_model.encode_image(images).float(), dim=-1)
        img_e = self.img_head(img_f)

        if captions is not None and CFG["USE_CAPTION"]:
            txt_f = self.encode_text_batch(captions)
            txt_e = self.txt_head(txt_f)
            gate  = torch.sigmoid(self.text_gate)
            fused = img_e + gate * txt_e
        else:
            fused = img_e

        return self.classifier(fused)


print("CLIPFusionClassifier defined ✅")

CLIPFusionClassifier defined ✅


In [12]:
# CELL 8 — Train & Validate Utilities (updated for MixUp/CutMix + captions)

def train_one_epoch(model, loader, optimizer, scheduler, scaler, criterion, soft_criterion,
                    num_classes, mixup_alpha, cutmix_alpha, mixup_prob):
    model.train()
    total_loss, n = 0.0, len(loader)
    for i, (imgs, labels, captions) in enumerate(loader):
        imgs, labels = imgs.to(device), labels.to(device)

        # ── MixUp / CutMix ──────────────────────────────────────────────────
        use_mix = random.random() < mixup_prob
        if use_mix:
            y_onehot = F.one_hot(labels, num_classes).float()
            if random.random() < 0.5:
                imgs, soft_y = mixup_data(imgs, y_onehot, mixup_alpha)
            else:
                imgs, soft_y = cutmix_data(imgs, y_onehot, cutmix_alpha)

        optimizer.zero_grad()
        with autocast():
            logits = model(imgs, captions)
            if use_mix:
                loss = soft_criterion(logits, soft_y)
            else:
                loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total_loss += loss.item()
        if (i+1) % 50 == 0 or (i+1) == n:
            print(f"  [{i+1:4d}/{n}] loss={total_loss/(i+1):.4f}  lr={scheduler.get_last_lr()[0]:.2e}")
    return total_loss / n


@torch.no_grad()
def validate(model, loader, criterion):
    model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []
    for imgs, labels, captions in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        with autocast():
            logits = model(imgs, captions)
            loss   = criterion(logits, labels)
        total_loss += loss.item()
        all_preds  += logits.argmax(1).cpu().tolist()
        all_labels += labels.cpu().tolist()
    f1  = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    acc = np.mean(np.array(all_preds) == np.array(all_labels))
    return total_loss / len(loader), acc, f1


@torch.no_grad()
def predict_probs(model, df, transform, batch_size, num_workers):
    model.eval()
    ds     = PanelDataset(df, transform, has_label=False)
    loader = DataLoader(ds, batch_size=batch_size*2, shuffle=False,
                        num_workers=num_workers, pin_memory=True)
    probs = []
    for imgs, captions in tqdm(loader, desc="Inference"):
        with autocast():
            logits = model(imgs.to(device), captions)
        probs.append(F.softmax(logits, dim=1).cpu().numpy())
    return np.concatenate(probs)


print("Utilities defined ✅")

Utilities defined ✅


In [ ]:
# CELL 9 — Stage 1: Train on TRAIN, validate on DEV

gc.collect(); torch.cuda.empty_cache()
seed_everything(CFG["SEED"])

clip_backbone, _, clip_preprocess = open_clip.create_model_and_transforms(
    CFG["CLIP_MODEL"], pretrained=CFG["CLIP_PRETRAINED"])

model = CLIPFusionClassifier(
    clip_backbone, num_classes,
    dropout=CFG["DROPOUT"],
    caption_weight=CFG["CAPTION_WEIGHT"]
).to(device)

# Freeze all CLIP, unfreeze last N visual blocks + projection
for p in model.clip_model.parameters():
    p.requires_grad = False
N = CFG["UNFREEZE_BLOCKS"]
for p in model.clip_model.visual.transformer.resblocks[-N:].parameters():
    p.requires_grad = True
if hasattr(model.clip_model.visual, 'proj') and model.clip_model.visual.proj is not None:
    model.clip_model.visual.proj.requires_grad = True
for p in model.img_head.parameters(): p.requires_grad = True
for p in model.txt_head.parameters(): p.requires_grad = True
for p in model.classifier.parameters(): p.requires_grad = True
model.text_gate.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable/1e6:.1f}M / {total/1e6:.1f}M")

train_tfm = get_transforms(CFG["IMG_SIZE"], "train")
val_tfm   = get_transforms(CFG["IMG_SIZE"], "val")

train_ds = PanelDataset(train_df, train_tfm)
dev_ds   = PanelDataset(dev_df_valid, val_tfm)

sampler      = make_weighted_sampler(train_df, class_weights)
train_loader = DataLoader(train_ds, batch_size=CFG["BATCH_SIZE"],
                          sampler=sampler,
                          num_workers=CFG["NUM_WORKERS"], pin_memory=True, drop_last=True)
dev_loader   = DataLoader(dev_ds, batch_size=CFG["BATCH_SIZE"]*2, shuffle=False,
                          num_workers=CFG["NUM_WORKERS"], pin_memory=True)

criterion      = CombinedLoss(gamma=CFG["FOCAL_GAMMA"], smoothing=CFG["LABEL_SMOOTH"],
                               weight=class_weights_tensor, focal_w=0.7)
soft_criterion = SoftCombinedLoss(gamma=CFG["FOCAL_GAMMA"], weight=class_weights_tensor)

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=CFG["LR"], weight_decay=CFG["WEIGHT_DECAY"])
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=CFG["LR"],
    total_steps=CFG["EPOCHS"] * len(train_loader),
    pct_start=CFG["WARMUP_EPOCHS"] / CFG["EPOCHS"],
    anneal_strategy="cos",
    final_div_factor=CFG["LR"] / CFG["MIN_LR"],
)
scaler = GradScaler()
es     = EarlyStopping(patience=CFG["PATIENCE"], save_path=CFG["SAVE_PATH"])

print(f"\n{'='*60}")
print(f"Stage 1: TRAIN → validate on DEV")
print(f"Train: {len(train_df)} | Dev: {len(dev_df_valid)} | Epochs: {CFG['EPOCHS']}")
print(f"Unfreeze: {N} blocks | MixUp+CutMix p={CFG['MIXUP_PROB']} | Caption fusion: {CFG['USE_CAPTION']}")
print(f"{'='*60}")

for epoch in range(CFG["EPOCHS"]):
    print(f"\nEpoch {epoch+1}/{CFG['EPOCHS']}")
    tl = train_one_epoch(
        model, train_loader, optimizer, scheduler, scaler,
        criterion, soft_criterion,
        num_classes, CFG["MIXUP_ALPHA"], CFG["CUTMIX_ALPHA"], CFG["MIXUP_PROB"]
    )
    vl, va, vf = validate(model, dev_loader, criterion)
    print(f"  train={tl:.4f}  val={vl:.4f}  acc={va:.4f}  F1={vf:.4f}")
    if es.step(va, vf, model):
        break

model.load_state_dict(torch.load(CFG["SAVE_PATH"], map_location=device))
print(f"\n✅ Stage-1 Best DEV Acc: {es.best_acc:.4f} | F1: {es.best_f1:.4f}")

100%|████████████████████████████████████████| 934M/934M [00:03<00:00, 247MiB/s]


Trainable: 152.9M / 428.9M

Stage 1: TRAIN → validate on DEV
Train: 2422 | Dev: 362 | Epochs: 25
Unfreeze: 12 blocks | MixUp+CutMix p=0.5 | Caption fusion: True

Epoch 1/25
  [  50/151] loss=21.2995  lr=3.12e-06
  [ 100/151] loss=17.8821  lr=8.36e-06
  [ 150/151] loss=15.5728  lr=1.55e-05
  [ 151/151] loss=15.4892  lr=1.57e-05
  train=15.4892  val=3.0868  acc=0.0912  F1=0.0578
  ✅ Best saved (Acc=0.0912, F1=0.0578)

Epoch 2/25
  [  50/151] loss=7.2708  lr=2.28e-05
  [ 100/151] loss=5.5093  lr=2.81e-05
  [ 150/151] loss=4.2573  lr=3.00e-05
  [ 151/151] loss=4.2353  lr=3.00e-05
  train=4.2353  val=2.7005  acc=0.2541  F1=0.1605
  ✅ Best saved (Acc=0.2541, F1=0.1605)

Epoch 3/25
  [  50/151] loss=1.1836  lr=3.00e-05
  [ 100/151] loss=1.0791  lr=2.99e-05
  [ 150/151] loss=1.0208  lr=2.99e-05
  [ 151/151] loss=1.0195  lr=2.99e-05
  train=1.0195  val=2.7371  acc=0.4890  F1=0.2808
  ✅ Best saved (Acc=0.4890, F1=0.2808)

Epoch 4/25
  [  50/151] loss=0.8011  lr=2.98e-05
  [ 100/151] loss=0.7284 

In [14]:
# CELL 10 — DEV Evaluation + Temperature Scaling
# Temperature scaling: finds T* that maximises dev accuracy.
# Calibrated probs → more reliable TTA averaging later.

gc.collect(); torch.cuda.empty_cache()

dev_probs_raw = predict_probs(model, dev_df_valid, val_tfm, CFG["BATCH_SIZE"], CFG["NUM_WORKERS"])
dev_preds_raw = dev_probs_raw.argmax(axis=1)
dev_gt        = dev_df_valid["label_id"].values

dev_acc_raw = np.mean(dev_preds_raw == dev_gt)
dev_f1_raw  = f1_score(dev_gt, dev_preds_raw, average="macro", zero_division=0)
print(f"Stage-1 DEV  Acc: {dev_acc_raw:.4f}  |  F1: {dev_f1_raw:.4f}")

# ── Temperature scaling (find optimal T on dev) ──────────────────────────────
# Convert raw probs back to logit-scale, then sweep T
log_probs = np.log(dev_probs_raw + 1e-10)   # log-softmax approx

best_T, best_acc_T = 1.0, dev_acc_raw
for T in np.arange(0.5, 2.0, 0.05):
    scaled = np.exp(log_probs / T)
    scaled /= scaled.sum(axis=1, keepdims=True)
    preds_T = scaled.argmax(axis=1)
    acc_T   = np.mean(preds_T == dev_gt)
    if acc_T > best_acc_T:
        best_acc_T = acc_T
        best_T     = T

print(f"Temperature scaling: T*={best_T:.2f} → DEV Acc={best_acc_T:.4f} "
      f"(gain +{best_acc_T-dev_acc_raw:.4f})")

# ── Per-class report ─────────────────────────────────────────────────────────
present_ids   = sorted(np.unique(dev_gt))
present_names = le.inverse_transform(present_ids)
print("\nPer-class breakdown:")
print(classification_report(dev_gt, dev_preds_raw,
                             labels=present_ids, target_names=present_names, digits=3))

Inference: 100%|██████████| 12/12 [00:14<00:00,  1.20s/it]

Stage-1 DEV  Acc: 0.7238  |  F1: 0.5029
Temperature scaling: T*=1.00 → DEV Acc=0.7238 (gain +0.0000)

Per-class breakdown:
                             precision    recall  f1-score   support

          apparatus diagram      0.143     1.000     0.250         1
                  bar chart      1.000     1.000     1.000         2
         conceptual diagram      0.611     0.314     0.415        35
   device structure diagram      0.000     0.000     0.000         2
                image panel      0.767     1.000     0.868        33
                 line chart      0.727     0.552     0.627        29
molecular structure diagram      0.952     0.980     0.966       101
        multi spectra chart      0.409     0.900     0.562        10
           multi-axis chart      0.800     0.800     0.800         5
        multiple line chart      0.818     0.551     0.659        49
      multiple scatter plot      0.500     0.667     0.571        15
       process flow diagram      0.133     1.000

In [ ]:
# CELL 11 — Pseudo-labeling on Test Set (threshold lowered to 0.85)
gc.collect(); torch.cuda.empty_cache()

print("Running inference on test set for pseudo-labels...")
test_probs_s1 = predict_probs(model, test_df, val_tfm, CFG["BATCH_SIZE"], CFG["NUM_WORKERS"])

test_confidence = test_probs_s1.max(axis=1)
test_pred_ids   = test_probs_s1.argmax(axis=1)

pseudo_mask = test_confidence >= CFG["PSEUDO_THRESH"]
print(f"\nTest samples          : {len(test_df)}")
print(f"Pseudo-label threshold: {CFG['PSEUDO_THRESH']}  (v14=0.90)")
print(f"High-confidence       : {pseudo_mask.sum()} ({100*pseudo_mask.mean():.1f}%)")

pseudo_df = test_df[pseudo_mask].copy().reset_index(drop=True)
pseudo_df["label_id"] = test_pred_ids[pseudo_mask]
pseudo_df["label"]    = le.inverse_transform(pseudo_df["label_id"])

print(f"\nPseudo-label class distribution (top 10):")
for cls, cnt in Counter(pseudo_df["label"]).most_common(10):
    print(f"  {cnt:4d}  {cls}")

rare_classes = [le.classes_[i] for i in range(num_classes) if class_counts[i] < 10]
pseudo_rare  = pseudo_df[pseudo_df["label"].isin(rare_classes)]
print(f"\nPseudo-labels for rare classes (<10 train samples): {len(pseudo_rare)}")

In [19]:
gc.collect(); torch.cuda.empty_cache()

In [ ]:
# CELL 12 — Stage 2: Full fine-tune on TRAIN+DEV+PSEUDO
import gc
gc.collect()
torch.cuda.empty_cache()

seed_everything(CFG["SEED"] + 1)

# 🔧 MEMORY SETTINGS
CFG["BATCH_SIZE"] = 8
CFG["FT_UNFREEZE_ALL"] = False
CFG["UNFREEZE_BLOCKS"] = 2
ACCUM_STEPS = 4

# 📊 Data prep
full_pool = pd.concat([train_df, dev_df_valid, pseudo_df], ignore_index=True)
print(f"Full pool: {len(train_df)} train + {len(dev_df_valid)} dev + {len(pseudo_df)} pseudo = {len(full_pool)} total")

label_counts_pool = Counter(full_pool["label_id"])
rare_mask  = full_pool["label_id"].map(lambda x: label_counts_pool[x] < 2)
rare_pool  = full_pool[rare_mask]
split_pool = full_pool[~rare_mask]

if len(split_pool) > 0:
    s2_train_split, s2_val_split = train_test_split(
        split_pool, test_size=0.10,
        stratify=split_pool["label_id"],
        random_state=CFG["SEED"]
    )
    s2_train_df = pd.concat([s2_train_split, rare_pool], ignore_index=True)
    s2_val_df   = s2_val_split.reset_index(drop=True)
else:
    s2_train_df = full_pool.copy()
    s2_val_df   = dev_df_valid.copy()

print(f"Stage-2 train: {len(s2_train_df)} | Stage-2 val: {len(s2_val_df)}")

# ⚖️ Class weights
cw_s2 = compute_class_weight("balanced", classes=np.arange(num_classes),
                            y=s2_train_df["label_id"].values)
cw_s2_tensor = torch.tensor(cw_s2, dtype=torch.float).to(device)

# 🔁 Load model
clip_bb2, _, _ = open_clip.create_model_and_transforms(
    CFG["CLIP_MODEL"], pretrained=CFG["CLIP_PRETRAINED"])

model_full = CLIPFusionClassifier(
    clip_bb2, num_classes,
    dropout=CFG["DROPOUT"],
    caption_weight=CFG["CAPTION_WEIGHT"]
).to(device)

model_full.load_state_dict(torch.load(CFG["SAVE_PATH"], map_location=device))

# ✅ Gradient checkpointing
model_full.clip_model.visual.set_grad_checkpointing(True)

# 🔓 Partial unfreezing
for p in model_full.clip_model.parameters():
    p.requires_grad = False

N = CFG["UNFREEZE_BLOCKS"]
for p in model_full.clip_model.visual.transformer.resblocks[-N:].parameters():
    p.requires_grad = True

if hasattr(model_full.clip_model.visual, 'proj') and model_full.clip_model.visual.proj is not None:
    model_full.clip_model.visual.proj.requires_grad = True

# heads trainable
for p in model_full.img_head.parameters(): p.requires_grad = True
for p in model_full.txt_head.parameters(): p.requires_grad = True
for p in model_full.classifier.parameters(): p.requires_grad = True
model_full.text_gate.requires_grad = True

trainable2 = sum(p.numel() for p in model_full.parameters() if p.requires_grad)
total2     = sum(p.numel() for p in model_full.parameters())
print(f"Stage-2 Trainable: {trainable2/1e6:.1f}M / {total2/1e6:.1f}M")

# 📦 Data loaders
s2_sampler = make_weighted_sampler(s2_train_df, cw_s2)
ft_tfm     = get_transforms(CFG["IMG_SIZE"], "train")

ft_ds     = PanelDataset(s2_train_df, ft_tfm)
ft_val_ds = PanelDataset(s2_val_df, val_tfm)

ft_loader = DataLoader(
    ft_ds,
    batch_size=CFG["BATCH_SIZE"],
    sampler=s2_sampler,
    num_workers=CFG["NUM_WORKERS"],
    pin_memory=True,
    drop_last=True
)

ft_val_loader = DataLoader(
    ft_val_ds,
    batch_size=CFG["BATCH_SIZE"] * 2,
    shuffle=False,
    num_workers=CFG["NUM_WORKERS"],
    pin_memory=True
)

# 🎯 Loss + optimizer
crit_ft = CombinedLoss(
    gamma=CFG["FOCAL_GAMMA"],
    smoothing=CFG["LABEL_SMOOTH"],
    weight=cw_s2_tensor,
    focal_w=0.7
)

soft_crit_ft = SoftCombinedLoss(
    gamma=CFG["FOCAL_GAMMA"],
    weight=cw_s2_tensor
)

opt_ft = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model_full.parameters()),
    lr=CFG["FT_LR"],
    weight_decay=CFG["WEIGHT_DECAY"]
)

sch_ft = torch.optim.lr_scheduler.OneCycleLR(
    opt_ft,
    max_lr=CFG["FT_LR"],
    total_steps=CFG["FT_EPOCHS"] * len(ft_loader) // ACCUM_STEPS,
    pct_start=1/CFG["FT_EPOCHS"],
    anneal_strategy="cos",
    final_div_factor=CFG["FT_LR"] / CFG["MIN_LR"],
)

sc_ft = GradScaler()
es_ft = EarlyStopping(patience=CFG["FT_PATIENCE"], save_path=CFG["SAVE_PATH_FULL"])

# 🚀 TRAIN LOOP
print(f"\n── Stage 2: up to {CFG['FT_EPOCHS']} epochs ──")

for epoch in range(CFG["FT_EPOCHS"]):
    print(f"\nEpoch {epoch+1}/{CFG['FT_EPOCHS']}")

    model_full.train()
    total_loss = 0
    opt_ft.zero_grad()

    for step, batch in enumerate(ft_loader):

        # ✅ FIXED: correct unpacking
        imgs, labels, captions = batch

        imgs   = imgs.to(device)
        labels = labels.to(device)

        with autocast():
            outputs = model_full(imgs, captions)
            loss = crit_ft(outputs, labels)

        loss = loss / ACCUM_STEPS
        sc_ft.scale(loss).backward()

        if (step + 1) % ACCUM_STEPS == 0:
            sc_ft.step(opt_ft)
            sc_ft.update()
            opt_ft.zero_grad()
            sch_ft.step()

        total_loss += loss.item()

    tl = total_loss / len(ft_loader)

    # 🔍 Validation (IMPORTANT: also uses captions)
    model_full.eval()
    vl, va, vf = validate(model_full, ft_val_loader, crit_ft)

    print(f"  train={tl:.4f}  val={vl:.4f}  acc={va:.4f}  F1={vf:.4f}")

    if es_ft.step(va, vf, model_full):
        break

# ✅ Load best model
model_full.load_state_dict(torch.load(CFG["SAVE_PATH_FULL"], map_location=device))

print(f"\n✅ Stage-2 Best Val Acc: {es_ft.best_acc:.4f} | F1: {es_ft.best_f1:.4f}")

Full pool: 2422 train + 362 dev + 853 pseudo = 3637 total
Stage-2 train: 3273 | Stage-2 val: 364
Stage-2 Trainable: 26.9M / 428.9M

── Stage 2: up to 15 epochs ──

Epoch 1/15
  train=0.6429  val=0.3294  acc=0.9505  F1=0.9486
  ✅ Best saved (Acc=0.9505, F1=0.9486)

Epoch 2/15
  train=0.1206  val=0.3171  acc=0.9478  F1=0.9123
  ⏳ No improvement (1/6)  best_acc=0.9505

Epoch 3/15
  train=0.0872  val=0.3010  acc=0.9478  F1=0.9152
  ⏳ No improvement (2/6)  best_acc=0.9505

Epoch 4/15
  train=0.0760  val=0.2921  acc=0.9368  F1=0.9082
  ⏳ No improvement (3/6)  best_acc=0.9505

Epoch 5/15
  train=0.0703  val=0.2913  acc=0.9313  F1=0.8976
  ⏳ No improvement (4/6)  best_acc=0.9505

Epoch 6/15
  train=0.0669  val=0.2843  acc=0.9423  F1=0.9234
  ⏳ No improvement (5/6)  best_acc=0.9505

Epoch 7/15
  train=0.0615  val=0.2825  acc=0.9478  F1=0.9551
  ⏳ No improvement (6/6)  best_acc=0.9505
  🛑 Early stopping triggered

✅ Stage-2 Best Val Acc: 0.9505 | F1: 0.9486


In [21]:
# CELL 13 — Test Inference: Extended TTA + Stage1/Stage2 Ensemble
# NEW vs v14:
#   - 8 TTA augments (was 6): added CLAHE and Sharpen variants
#   - Ensemble Stage1 + Stage2 probs (each with full TTA)
#   - Temperature scaling applied to all prob outputs

gc.collect(); torch.cuda.empty_cache()

norm = dict(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
sz = CFG["IMG_SIZE"]

tta_tfms = [
    # 1. Original
    A.Compose([A.Resize(sz, sz), A.Normalize(**norm), ToTensorV2()]),
    # 2. Horizontal flip
    A.Compose([A.Resize(sz, sz), A.HorizontalFlip(p=1), A.Normalize(**norm), ToTensorV2()]),
    # 3. Slight zoom
    A.Compose([A.Resize(int(sz*1.1), int(sz*1.1)), A.CenterCrop(sz, sz),
               A.Normalize(**norm), ToTensorV2()]),
    # 4. Slight zoom + flip
    A.Compose([A.Resize(int(sz*1.1), int(sz*1.1)), A.CenterCrop(sz, sz),
               A.HorizontalFlip(p=1), A.Normalize(**norm), ToTensorV2()]),
    # 5. Brightness/contrast
    A.Compose([A.Resize(sz, sz), A.RandomBrightnessContrast(0.1, 0.1, p=1),
               A.Normalize(**norm), ToTensorV2()]),
    # 6. Brightness/contrast + flip
    A.Compose([A.Resize(sz, sz), A.HorizontalFlip(p=1),
               A.RandomBrightnessContrast(0.1, 0.1, p=1),
               A.Normalize(**norm), ToTensorV2()]),
    # 7. NEW: CLAHE (enhances low-contrast scientific figures)
    A.Compose([A.Resize(sz, sz), A.CLAHE(p=1), A.Normalize(**norm), ToTensorV2()]),
    # 8. NEW: Sharpen (enhances text/axes in charts)
    A.Compose([A.Resize(sz, sz), A.Sharpen(p=1), A.Normalize(**norm), ToTensorV2()]),
]


def run_tta(model, df, tta_transforms, batch_size, num_workers, temperature=1.0):
    """Run full TTA for one model, return averaged calibrated probs."""
    model.eval()
    tta_probs = []
    for t, tfm in enumerate(tta_transforms):
        ds     = PanelDataset(df, tfm, has_label=False)
        loader = DataLoader(ds, batch_size=batch_size*2, shuffle=False,
                            num_workers=num_workers, pin_memory=True)
        probs = []
        with torch.no_grad():
            for imgs, captions in loader:
                with autocast():
                    logits = model(imgs.to(device), captions)
                # Apply temperature scaling before softmax
                probs.append(F.softmax(logits / temperature, dim=1).cpu().numpy())
        tta_probs.append(np.concatenate(probs))
        print(f"  TTA {t+1}/{len(tta_transforms)} done")
    return np.mean(tta_probs, axis=0)


print("── Stage-1 TTA inference ──")
probs_s1 = run_tta(model, test_df, tta_tfms,
                   CFG["BATCH_SIZE"], CFG["NUM_WORKERS"],
                   temperature=best_T)

print("\n── Stage-2 TTA inference ──")
probs_s2 = run_tta(model_full, test_df, tta_tfms,
                   CFG["BATCH_SIZE"], CFG["NUM_WORKERS"],
                   temperature=best_T)

# NEW: ensemble Stage1 + Stage2 (equal weight — adjust if one is consistently better)
final_probs  = 0.4 * probs_s1 + 0.6 * probs_s2   # Stage2 weighted slightly higher
final_preds  = final_probs.argmax(axis=1)
final_labels = le.inverse_transform(final_preds)
test_df["pred_label"] = final_labels

print(f"\nTest predictions: {len(test_df)}")
print(f"Top predicted classes: {Counter(final_labels).most_common(5)}")
print(f"\nStage1 TTA + Stage2 TTA ensemble → submitted")

── Stage-1 TTA inference ──
  TTA 1/8 done
  TTA 2/8 done
  TTA 3/8 done
  TTA 4/8 done
  TTA 5/8 done
  TTA 6/8 done
  TTA 7/8 done
  TTA 8/8 done

── Stage-2 TTA inference ──
  TTA 1/8 done
  TTA 2/8 done
  TTA 3/8 done
  TTA 4/8 done
  TTA 5/8 done
  TTA 6/8 done
  TTA 7/8 done
  TTA 8/8 done

Test predictions: 1121
Top predicted classes: [('molecular structure diagram', 251), ('multiple line chart', 130), ('multi spectra chart', 101), ('multiple scatter plot', 91), ('image panel', 83)]

Stage1 TTA + Stage2 TTA ensemble → submitted


In [ ]:
# CELL 14 — Build & Validate Submission File

sample_panels = defaultdict(dict)
for _, row in test_df.iterrows():
    sample_panels[row["sample_id"]][row["panel"]] = row["pred_label"]

sample_bboxes = {}
for jp in test_jsons:
    try:
        with open(jp) as f:
            data = json.load(f)
        if "bbox" in data:
            sample_bboxes[data["sample_id"]] = data["bbox"]
    except Exception:
        continue

results = []
for sample_id, panel_preds in sample_panels.items():
    bbox = sample_bboxes.get(sample_id, {})
    for p in bbox:
        if p not in panel_preds:
            panel_preds[p] = "unknown"
    results.append({"sample_id": sample_id, "bbox": bbox, "classification": panel_preds})

errors = []
for entry in results:
    for panel in entry.get("bbox", {}):
        if panel not in entry["classification"]:
            errors.append(f"{entry['sample_id']}: panel '{panel}' missing")

if errors:
    print("❌ Validation FAILED:")
    for e in errors: print(" ", e)
else:
    print(f"✅ Validation passed — {len(results)} samples")

print("\n--- FORMAT PREVIEW ---")
print(json.dumps(results[0], indent=2))

with open("prediction_data.json", "w") as f:
    json.dump(results, f, indent=2)
with zipfile.ZipFile("prediction_data.zip", "w", zipfile.ZIP_DEFLATED) as z:
    z.write("prediction_data.json", arcname="prediction_data.json")

print(f"\n✅ prediction_data.zip ready — {len(results)} samples")

✅ Validation passed — 580 samples

--- FORMAT PREVIEW ---
{
  "sample_id": "atomic-layer-deposition/simulation-usecase/20/fig_5",
  "bbox": {
    "a": {
      "x": 3,
      "y": 5,
      "width": 659,
      "height": 463
    },
    "b": {
      "x": 0,
      "y": 472,
      "width": 665,
      "height": 452
    }
  },
  "classification": {
    "a": "multi-axis chart",
    "b": "multi-axis chart"
  }
}

✅ prediction_data.zip ready — 580 samples
